In [1]:
import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [2]:
%%writefile requirements.txt

nltk>=3.0.0
notebook>=7.0.0
seaborn
torch>=2.0.0
torchvision
tqdm>=4.0.0

Writing requirements.txt


In [3]:
!pip install -r requirements.txt

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 89.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 914.5/914.5 kB 50.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 393.1/393.1 kB 26.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.2/17.2 MB 87.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.5/77.5 kB 6.3 MB/s eta 0:00:00
  Attempting uninstall: jupyter-server
    Found existing installation: jupyter_server 2.12.5
    Uninstalling jupyter_server-2.12.5:
      Successfully uninstalled jupyter_server-2.12.5
  Attempting uninstall: jupyter-lsp
    Found existing installation: jupyter-lsp 1.5.1
    Uninstalling jupyter-lsp-1.5.1:
      Successfully uninstalled jupyter-lsp-1.5.1
  Attempting uninstall: jupyterlab
    Found existing installation: jupyterlab 3.6.8
    Uninstalling jupyterlab-3.6.8:
      Successfully uninstalled jupyterlab-3.6.8
  Attempting uninstall: notebook
    Found existing installation: notebook 6.

In [4]:
%%writefile utils.py

from torch import Tensor, optim


class AverageMeter:
    """
    跟踪指标的最近值、平均值、总和和计数
    """

    def __init__(self):
        self.reset()

    def reset(self):
        self.val = 0
        self.avg = 0
        self.sum = 0
        self.count = 0

    def update(self, val, n=1):
        self.val = val
        self.sum += val * n
        self.count += n
        self.avg = self.sum / self.count


def adjust_learning_rate(optimizer: optim.Optimizer, shrink_factor: float):
    """
    按指定因子缩小学习率

    :param optimizer: 学习率需要缩小的优化器
    :param shrink_factor: 乘以学习率的因子，在区间(0, 1)内
    """
    for param_group in optimizer.param_groups:
        param_group["lr"] *= shrink_factor


def accuracy(scores: Tensor, targets: Tensor, k: int) -> float:
    """
    计算top-k准确率

    :param scores: 模型的输出分数
    :param targets: 真实标签
    :param k: top-k准确率中的k
    :return: top-k准确率
    """

    batch_size = targets.size(0)
    _, ind = scores.topk(k, 1, True, True)
    correct = ind.eq(targets.view(-1, 1).expand_as(ind))
    correct_total = correct.view(-1).float().sum()  # 0D张量
    return correct_total.item() * (100.0 / batch_size)


Writing utils.py


In [5]:
%%writefile caption_dataset.py

import json
import os
import typing
from pathlib import Path
from typing import Literal

import torch
from PIL import Image
from torch import Tensor
from torch.utils.data import Dataset
from tqdm import tqdm

PAD_TOKEN = 0
DatasetSplit = Literal["train", "val", "test"]


def collate_fn(batch: list[tuple]):
    """
    Collate function for CaptionDataset.
    """
    is_train = len(batch[0]) == 2

    if is_train:
        imgs, captions = zip(*batch)
    else:
        imgs, captions, all_captions = zip(*batch)

    if isinstance(imgs[0], Tensor):
        imgs = torch.stack(imgs, 0)

    lengths = [len(cap) for cap in captions]
    max_length = max(lengths)
    padded_captions = torch.full(
        (len(captions), max_length), PAD_TOKEN, dtype=torch.long
    )
    for i, cap in enumerate(captions):
        end = lengths[i]
        padded_captions[i, :end] = torch.tensor(cap[:end], dtype=torch.long)
    lengths = torch.tensor(lengths)

    if is_train:
        return imgs, padded_captions, lengths
    else:
        return imgs, padded_captions, lengths, all_captions


class CaptionDataset(Dataset):
    """Load preprocessed caption dataset."""

    def __init__(
        self,
        root: str | Path,
        preprocess_dir: str | Path, 
        split: DatasetSplit = "train",
        transform=None,
        cache=False,
        max_cap_len=-1,
    ):
        super().__init__()
        self.split = split
        assert self.split in typing.get_args(DatasetSplit)

        root = Path(root)
        preprocess_dir = Path(preprocess_dir) 

        self.root = root
        self.preprocess_dir = preprocess_dir

        if not root.is_dir():
            raise RuntimeError(f"Image root does not exist: {root}")

        self.img_dir = root / "Images"
        if not self.img_dir.is_dir():
            self.img_dir = root / "images"

        # Note on Load, File. Refer to the archived source for the detailed original comment.
        enc_path = preprocess_dir / "captions_encoded.json"
        vocab_path = preprocess_dir / "vocab.json"
        if not enc_path.exists():
            raise FileNotFoundError(f"Missing {enc_path}")
        if not vocab_path.exists():
            raise FileNotFoundError(f"Missing {vocab_path}")

        with open(enc_path, "r") as f:
            captions: list[dict] = json.load(f)
        with open(vocab_path, "r") as f:
            self.vocab: dict[str, int] = json.load(f)

        self.rev_vocab: dict[int, str] = {i: w for w, i in self.vocab.items()}

        # Note on Data. Refer to the archived source for the detailed original comment.
        self.img_names: list[str] = []
        self.captions: list[dict] = []
        max_cap_len_in_data = 0
        for caption in captions:
            if caption["split"] != self.split:
                continue
            img_idx = len(self.img_names)
            self.img_names.append(caption["filename"])
            for sentence in caption["sentences"]:
                max_cap_len_in_data = max(max_cap_len_in_data, len(sentence))
                self.captions.append({"img_idx": img_idx, "caption": sentence})
        self.max_cap_len = max_cap_len_in_data if max_cap_len <= 0 else max_cap_len
        self.cpi = len(captions[0]["sentences"]) if captions else 5
        self.transform = transform

        # Note on Image. Refer to the archived source for the detailed original comment.
        self.img_cache = []
        if cache:
            cache_dir = preprocess_dir / "cache" / self.split
            cache_dir.mkdir(parents=True, exist_ok=True)
            for name in tqdm(self.img_names, desc=f"Caching {self.split} images"):
                img = Image.open(self.img_dir / name).convert("RGB")
                if self.transform:
                    img = self.transform(img)
                self.img_cache.append(img)

    def __getitem__(self, idx: int):
        if self.img_cache:
            img = self.img_cache[self.captions[idx]["img_idx"]]
        else:
            img = Image.open(self.img_dir / self.img_names[self.captions[idx]["img_idx"]]).convert("RGB")
            if self.transform:
                img = self.transform(img)
        caption: list[int] = self.captions[idx]["caption"]

        if self.split == "train":
            return img, caption
        else:
            start_idx = (idx // self.cpi) * self.cpi
            end_idx = start_idx + self.cpi
            all_captions = [cap["caption"] for cap in self.captions[start_idx:end_idx]]
            return img, caption, all_captions

    def __len__(self):
        return len(self.captions)

Writing caption_dataset.py


### Model

This cell documents the image caption generation experiment. For the full original methodology, mathematical derivations, and commentary, see the archived notebook in `archive/original_notebooks/`.


In [6]:
!mkdir -p /kaggle/working/models

In [7]:
%%writefile models/encoder.py

import torch
import torchvision.models as models
from torch import Tensor, nn
from torchvision.models import ResNet50_Weights


class Encoder(nn.Module):
    """Encoder model."""

    def __init__(self, encoded_size=(7, 7), finetune=False):
        super().__init__()
        # Note on Training, Load. Refer to the archived source for the detailed original comment.
        resnet = models.resnet50(weights=ResNet50_Weights.DEFAULT)
        # 移除最后两层（平均池化和全连接层）
        self.resnet = nn.Sequential(*list(resnet.children())[:-2])
        # 自适应池化到固定尺寸
        self.pool = nn.AdaptiveAvgPool2d(encoded_size)
        # 默认不微调
        self.finetune(finetune)

    def forward(self, images: Tensor):
        """Extract image features.

        Args:
            images (Tensor): Input images with shape (batch, channels, height, width).

        Returns:
            Tensor: Extracted features with shape (batch, num_pixels, feature_dim).
        """
        # Note on Features. Refer to the archived source for the detailed original comment.
        x: Tensor = self.resnet(images)
        # 池化到固定尺寸
        x: Tensor = self.pool(x)
        # B C H W -> B (H W) C
        x = x.permute(0, 2, 3, 1).flatten(1, 2)
        return x

    def finetune(self, finetune=True):
        # Note on Parameters. Refer to the archived source for the detailed original comment.
        self.resnet.requires_grad_(False)
        # 仅微调后几层
        # for child in list(self.resnet.children())[5:]:
        #     child.requires_grad_(finetune)

        # 只微调最后三个残差块
        for name, child in self.resnet.named_children():
            if name in ['layer2', 'layer3', 'layer4']:
                child.requires_grad_(finetune)

        # 冻结部分保持 eval 模式（固定 BN/Dropout 等）
        if finetune:
            # Note on Training. Refer to the archived source for the detailed original comment.
            for name, child in self.resnet.named_children():
                if name in ['layer2', 'layer3', 'layer4']:
                    child.train()
                else:
                    child.eval()
        else:
            # 完全不微调时，整个 backbone 设为 eval
            self.resnet.eval()



Writing models/encoder.py


In [8]:
%%writefile models/decoder.py

import torch
from torch import Tensor, nn

from .attention import AdditiveAttention


class RNNDecoder(nn.Module):
    def __init__(
        self,
        embed_dim: int,
        hidden_dim: int,
        vocab_size: int,
        feature_dim: int,
        dropout=0.5,
        rnn_cell=nn.LSTMCell,
    ):
        """Initialize RNNDecoder.

        Args:
            embed_dim (int): Embedding dimension.
            hidden_dim (int): Hidden dimension.
            vocab_size (int): Vocabulary size.
            feature_dim (int): Feature dimension.
            dropout (float): Dropout rate. Default: 0.5.
            rnn_cell (Module): RNN cell module. Default: nn.LSTMCell.
        """
        super().__init__()
        self.hidden_dim = hidden_dim
        
        # ============ TODO: Init layers ============
        # Decoder expects these layers:
        # Note on Input. Refer to the archived source for the detailed original comment.
        # - lstm: lstm cell
        # - init_h: linear projection to init hidden state from image features
        #    Note on Image, Features. Refer to the archived source for the detailed original comment.
        # - init_c: linear projection to init cell state from image features
        #    Note on Image, Features. Refer to the archived source for the detailed original comment.
        # - dropout: output dropout layer
        # - fc: output linear projection全连接层
        # raise NotImplementedError
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.lstm = rnn_cell(embed_dim, hidden_dim)
        self.lstm_cell = self.lstm
        self.init_h = nn.Linear(feature_dim, hidden_dim)
        self.init_c = nn.Linear(feature_dim, hidden_dim)
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_dim, vocab_size)
        
        # ===========================================

        # initialize some layers with the uniform distribution
        self.embedding.weight.data.uniform_(-0.1, 0.1)
        self.fc.bias.data.fill_(0)
        self.fc.weight.data.uniform_(-0.1, 0.1)

    def init_hidden_state(self, features: Tensor) -> tuple[Tensor, Tensor]:
        """Initialize hidden states for RNNDecoder.

        Args:
            features (Tensor): Input features with shape (batch, num_pixels, feature_dim).

        Returns:
            tuple: (hidden_state, cell_state) both with shape (batch, hidden_dim).
        """
        # ========== TODO: Init hidden states ==========
        # raise NotImplementedError
        pooled_features = features.mean(dim=1)# 对像素维度取平均，使得 features 变为 (batch, feature_dim)
        h = self.init_h(pooled_features)   # (batch_size, hidden_size)
        c = self.init_c(pooled_features)
        # ==============================================
        return h, c

    def forward(self, features: Tensor, captions: Tensor):
        """Forward pass of RNNDecoder.

        Args:
            features (Tensor): Input features with shape (batch, num_pixels, feature_dim).
            captions (Tensor): Input captions with shape (batch, max_caption_length).

        Returns:
            Tensor: Output predictions with shape (batch, max_caption_length, vocab_size).
        """
        B = features.shape[0]
        T = captions.shape[1]

        # 词嵌入
        embeddings: Tensor = self.embedding(captions)
        # Note on Initialize. Refer to the archived source for the detailed original comment.
        h, c = self.init_hidden_state(features)

        outputs: list[Tensor] = []
        # Decode step by step
        for t in range(T - 1):
            # ======== TODO: implement decode step ========
            # Hint 1: apply dropout just before fc
            # Hint 2: use teacher-forcing here
            # raise NotImplementedError
            current_embed = embeddings[:, t, :] # 取当前时间步的词嵌入 (batch_size, embed_size)
            h, c = self.lstm_cell(current_embed, (h, c))   # 传入 LSTM Cell，更新 h 和 c
            h_dropped = self.dropout(h)   # 经过 Dropout
            
            logits = self.fc(h_dropped)
            outputs.append(logits)   # 堆叠所有时间步的输出，调整维度为 (batch_size, seq_len, vocab_size)
            # =============================================

        # Note on Output. Refer to the archived source for the detailed original comment.
        return torch.stack(outputs, dim=1)

    def generate_caption(
        self, features: Tensor, word_map: dict[str, int], max_len=30, beam_size=1
    ):
        """Generate captions with greedy decoding or beam search (during test time).

        Args:
            features (Tensor): Input features with shape (batch, num_pixels, feature_dim).
            word_map (dict[str, int]): Word map from token to index.
            max_len (int): Maximum caption length. Default: 30.
            beam_size (int): Beam size for beam search. Default: 1 (greedy decoding).

        Returns:
            list[list[int]]: Generated captions.
        """
        B = features.shape[0]
        features = features.view(B, -1, features.shape[-1])
        device = features.device

        # 获取特殊标记
        start_token = word_map["<start>"]
        end_token = word_map["<end>"]

        if beam_size == 1:
            # Note on Use. Refer to the archived source for the detailed original comment.
            # Note on Initialize. Refer to the archived source for the detailed original comment.
            h, c = self.init_hidden_state(features)

            input_tokens = torch.full(
                (B,),
                start_token,
                dtype=torch.long,
                device=device
            )

            captions = [[start_token] for _ in range(B)]
            finished = torch.zeros(B, dtype=torch.bool, device=device)

            # Note on Generate. Refer to the archived source for the detailed original comment.
            for _ in range(max_len - 1):
                # ======== TODO: implement generate step ========
                # Hint 1: apply dropout just before fc
                # Hint 2: a caption ends when it reaches the end token
                # 1. 获取当前 predictions
                # 2. 更新 captions
                # raise NotImplementedError
                current_embed = self.embedding(input_tokens) # 取当前时间步的词嵌入 (batch_size, embed_size)
                h, c = self.lstm_cell(current_embed, (h, c))   # 传入 LSTM Cell，更新 h 和 c  
                logits = self.fc(self.dropout(h))  # 经过 Dropout
                next_tokens = logits.argmax(dim=1)  # 贪心选择概率最高的词

                for i, token in enumerate(next_tokens.tolist()):
                    if not finished[i]:
                        captions[i].append(token)

                finished = finished | (next_tokens == end_token)
                input_tokens = next_tokens

                if finished.all():
                    break
            for i in range(B):
                if captions[i][-1] != end_token:
                    captions[i].append(end_token)
                
                # =============================================
            return captions
            
        else:
            # Note on Use. Refer to the archived source for the detailed original comment.
            captions: list[list[int]] = []
            # Note on Image. Refer to the archived source for the detailed original comment.
            for i in range(B):
                # Note on Image, Features. Refer to the archived source for the detailed original comment.
                img_features = features[i : i + 1]
                # Note on Initialize. Refer to the archived source for the detailed original comment.
                h, c = self.init_hidden_state(img_features)

                # ======== Optional TODO: implement beam search ========
                # Hint: you may need to store beam as (log_prob, hidden_state, cell_state, tokens)
                # 1. Initialize beam with start token
                # 2. Repeat for max_len - 1 steps:
                # 2.1. Compute log probabilities for all possible next tokens
                #      Note: compute only for non-terminated sequences
                # 2.2. Update beam with top-k (k=beam_size) candidates
                # 2.3. Sort beam by log probability, and keep top-k beams
                # 2.4. Stop if all sequences end with end token
                # 3. Select the sequence with highest log probability
                # raise NotImplementedError

                # beam 存储为 (log_prob, hidden, cell, sequence)
                beam = [(0.0, h, c, [])]
                
                for step in range(max_len - 1):
                    new_beam = []
                    # 对每个 beam 进行拓展
                    for log_prob, h_beam, c_beam, seq in beam:
                        # 如果当前序列已经以 <end> 结束，则直接保留
                        if seq and seq[-1] == end_token:
                            new_beam.append((log_prob, h_beam, c_beam, seq))
                            continue

                        # Note on Input. Refer to the archived source for the detailed original comment.
                        current_token = start_token if len(seq) == 0 else seq[-1]
                        current_tensor = torch.tensor([current_token], device=device).long()  # (1,)
                        embed = self.embedding(current_tensor)                                 # (1, embed_dim)
                
                        # 更新LSTM
                        h_new, c_new = self.lstm_cell(embed, (h_beam, c_beam)) 

                        # Note on Output. Refer to the archived source for the detailed original comment.
                        logits = self.fc(self.dropout(h_new))
                        log_probs = torch.log_softmax(logits, dim=1)

                        # 取 top-k（k = beam_size）个候选
                        top_log_probs, top_indices = log_probs.topk(beam_size, dim=1)

                        for k in range(beam_size):
                            cand_log_prob = log_prob + top_log_probs[0, k].item()
                            cand_token = top_indices[0, k].item()
                            new_seq = seq + [cand_token]
                            new_beam.append((cand_log_prob, h_new, c_new, new_seq))
                   
                    # 按 log_prob 降序排序，保留 beam_size 个候选
                    new_beam.sort(key=lambda x: x[0], reverse=True)
                    beam = new_beam[:beam_size]
                
                    # 如果所有 beam 都以 <end> 结尾，提前结束
                    if all(seq and seq[-1] == end_token for _, _, _, seq in beam):
                        break

                # 选择概率最高的序列
                best_seq = beam[0][3]
                full_seq = [start_token] + best_seq # 将 <start> 添加到开头
                
                # 若未以 <end> 结尾，添加（通常 beam 会在结束前停止，但为安全）
                if full_seq[-1] != end_token:
                    full_seq.append(end_token)
                    
                captions.append(full_seq)
                # ======================================================

            return captions


class RNNDecoderWithAttention(nn.Module):
    def __init__(
        self,
        embed_dim: int,
        hidden_dim: int,
        vocab_size: int,
        feature_dim: int,
        dropout=0.5,
        rnn_cell=nn.LSTMCell,
        attn_impl=AdditiveAttention,
    ):
        """Initialize RNNDecoder.

        Args:
            embed_dim (int): Embedding dimension.
            hidden_dim (int): Hidden dimension.
            vocab_size (int): Vocabulary size.
            feature_dim (int): Feature dimension.
            dropout (float): Dropout rate. Default: 0.5.
            rnn_cell (Module): RNN cell module. Default: nn.LSTMCell.
            attn_impl (Module): Attention module. Default: AdditiveAttention.
        """
        super().__init__()
        self.hidden_dim = hidden_dim
        # ============ TODO: Init layers ============
        # Decoder expects these layers:
        # - embedding: input embedding
        # - lstm: lstm cell
        # - attention: attention module
        # - init_h: linear projection to init hidden state from image features
        # - init_c: linear projection to init cell state from image features
        # - dropout: output dropout layer
        # - fc: output linear projection
        # raise NotImplementedError

        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.lstm = rnn_cell(embed_dim, hidden_dim)
        self.lstm_cell = self.lstm
        self.init_h = nn.Linear(feature_dim, hidden_dim)
        self.init_c = nn.Linear(feature_dim, hidden_dim)
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_dim, vocab_size)

        # Note on Note. Refer to the archived source for the detailed original comment.
        self.attention = AdditiveAttention(key_dim=feature_dim, query_dim=hidden_dim)
        self.att_embed = nn.Linear(feature_dim, embed_dim) # 将上下文向量投射到embed_dim, 以便与词嵌入相加
        # ===========================================

        # initialize some layers with the uniform distribution
        self.embedding.weight.data.uniform_(-0.1, 0.1)
        self.fc.bias.data.fill_(0)
        self.fc.weight.data.uniform_(-0.1, 0.1)

    def init_hidden_state(self, features: Tensor) -> tuple[Tensor, Tensor]:
        """Initialize hidden states for RNNDecoder.

        Args:
            features (Tensor): Input features with shape (batch, num_pixels, feature_dim).

        Returns:
            tuple: (hidden_state, cell_state) both with shape (batch, hidden_dim).
        """
        # ========== TODO: Init hidden states ==========
        # Same as RNNDecoder, you can copy the code
        # raise NotImplementedError
        pooled_features = features.mean(dim=1)# 对像素维度取平均，使得 features 变为 (batch, feature_dim)
        h = self.init_h(pooled_features)   # (batch_size, hidden_size)
        c = self.init_c(pooled_features)
        # ==============================================
        return h, c

    def forward(self, features: Tensor, captions: Tensor):
        """Forward pass of RNNDecoder.

        Args:
            features (Tensor): Input features with shape (batch, num_pixels, feature_dim).
            captions (Tensor): Input captions with shape (batch, max_caption_length).

        Returns:
            Tensor: Output predictions with shape (batch, max_caption_length, vocab_size).
        """
        B = features.shape[0]
        T = captions.shape[1]

        # 词嵌入
        embeddings = self.embedding(captions)
        # Note on Initialize. Refer to the archived source for the detailed original comment.
        h, c = self.init_hidden_state(features)

        outputs = []
        # 逐词解码
        for t in range(T - 1):
            # ======== TODO: implement decode step ========
            # Hint 1: apply dropout just before fc
            # Hint 2: use teacher-forcing here
            # raise NotImplementedError
            current_embed = embeddings[:, t, :] # 取当前时间步的词嵌入 (batch_size, embed_size)
            # Note on Image, Features. Refer to the archived source for the detailed original comment.
            context, alpha = self.attention(h, features)

            # 将 context 投影到 embed_dim 后与词嵌入相加
            context_proj = self.att_embed(context)
            lstm_input = current_embed + context_proj
            
            # Note on Compute. Refer to the archived source for the detailed original comment.
            h, c = self.lstm_cell(lstm_input, (h, c))   # 传入 LSTM Cell，更新 h 和 c
            
            logits = self.fc(self.dropout(h))
            outputs.append(logits)   # 堆叠所有时间步的输出，调整维度为 (batch_size, seq_len, vocab_size)
            # =============================================

        # Note on Output. Refer to the archived source for the detailed original comment.
        return torch.stack(outputs, dim=1)

    def generate_caption(
        self, features: Tensor, word_map: dict[str, int], max_len=30, beam_size=1
    ):
        """Generate captions with greedy decoding or beam search (during test time).

        Args:
            features (Tensor): Input features with shape (batch, num_pixels, feature_dim).
            word_map (dict[str, int]): Word map from token to index.
            max_len (int): Maximum caption length. Default: 30.
            beam_size (int): Beam size for beam search. Default: 1 (greedy decoding).

        Returns:
            list[list[int]]: Generated captions.
        """
        B = features.shape[0]
        features = features.view(B, -1, features.shape[-1])
        device = features.device

        # 获取特殊标记
        start_token = word_map["<start>"]
        end_token = word_map["<end>"]

        if beam_size == 1:
            # Note on Use. Refer to the archived source for the detailed original comment.
            # Note on Initialize. Refer to the archived source for the detailed original comment.
            h, c = self.init_hidden_state(features)

            input_tokens = torch.full(
                (B,),
                start_token,
                dtype=torch.long,
                device=device
            )

            captions = [[start_token] for _ in range(B)]
            finished = torch.zeros(B, dtype=torch.bool, device=device)

            # Note on Generate. Refer to the archived source for the detailed original comment.
            for _ in range(max_len - 1):
                # ======== TODO: implement generate step ========
                # Hint 1: apply dropout just before fc
                # Hint 2: a caption ends when it reaches the end token
                # 1. 获取当前 predictions
                # 2. 更新 captions
                # raise NotImplementedError
                current_embed = self.embedding(input_tokens) 

                # attention
                context, alpha = self.attention(h, features)
                context_proj = self.att_embed(context)
                lstm_input = current_embed + context_proj

                h, c = self.lstm_cell(lstm_input, (h, c))   # 更新 h 和 c  
                logits = self.fc(self.dropout(h))  # 经过 Dropout

                next_tokens = logits.argmax(dim=1)  # 贪心选择概率最高的词

                for i, token in enumerate(next_tokens.tolist()):
                    if not finished[i]:
                        captions[i].append(token)

                finished = finished | (next_tokens == end_token)
                input_tokens = next_tokens

                if finished.all():
                    break
                    
            for i in range(B):
                if captions[i][-1] != end_token:
                    captions[i].append(end_token)
                    
                # =============================================
            return captions

            
        else:
            # Note on Use. Refer to the archived source for the detailed original comment.
            captions: list[list[int]] = []
            # Note on Image. Refer to the archived source for the detailed original comment.
            for i in range(B):
                # Note on Image, Features. Refer to the archived source for the detailed original comment.
                img_features = features[i : i + 1]
                # Note on Initialize. Refer to the archived source for the detailed original comment.
                h, c = self.init_hidden_state(img_features)

                # ======== Optional TODO: implement beam search ========
                # Hint: you may need to store beam as (log_prob, hidden_state, cell_state, tokens)
                # 1. Initialize beam with start token
                # 2. Repeat for max_len - 1 steps:
                # 2.1. Compute log probabilities for all possible next tokens
                #      Note: compute only for non-terminated sequences
                # 2.2. Update beam with top-k (k=beam_size) candidates
                # 2.3. Sort beam by log probability, and keep top-k beams
                # 2.4. Stop if all sequences end with end token
                # 3. Select the sequence with highest log probability
                # raise NotImplementedError

                # beam 存储为 (log_prob, hidden, cell, sequence)
                beam = [(0.0, h, c, [])]
                
                for step in range(max_len - 1):
                    new_beam = []
                    # 对每个 beam 进行拓展
                    for log_prob, h_beam, c_beam, seq in beam:
                        # 如果当前序列已经以 <end> 结束，则直接保留
                        if seq and seq[-1] == end_token:
                            new_beam.append((log_prob, h_beam, c_beam, seq))
                            continue

                        current_token = start_token if len(seq) == 0 else seq[-1]
                        current_tensor = torch.tensor([current_token], device=device).long()
                        embed = self.embedding(current_tensor)

                        # Note on Note. Refer to the archived source for the detailed original comment.
                        context, _ = self.attention(h_beam, img_features)
                        context_proj = self.att_embed(context)
                        lstm_input = embed + context_proj
                
                        # 更新LSTM
                        h_new, c_new = self.lstm_cell(lstm_input, (h_beam, c_beam)) 

                        # Note on Output. Refer to the archived source for the detailed original comment.
                        h_dropped = self.dropout(h_new)   # 经过 Dropout
                        logits = self.fc(h_dropped)

                        log_probs = torch.log_softmax(logits, dim=1)

                        # 取 top-k（k = beam_size）个候选
                        top_log_probs, top_indices = log_probs.topk(beam_size, dim=1)

                        for k in range(beam_size):
                            cand_log_prob = log_prob + top_log_probs[0, k].item()
                            cand_token = top_indices[0, k].item()
                            new_seq = seq + [cand_token]
                            new_beam.append((cand_log_prob, h_new, c_new, new_seq))
        
                    # 排序并保留 top‑beam_size
                    new_beam.sort(key=lambda x: x[0], reverse=True)
                    beam = new_beam[:beam_size]

                    # 如果所有 beam 都以 <end> 结尾，提前结束
                    if all(seq and seq[-1] == end_token for _, _, _, seq in beam):
                        break

                # 选择概率最高的序列
                best_seq = beam[0][3]
                full_seq = [start_token] + best_seq # 将 <start> 添加到开头
                
                # 若未以 <end> 结尾，添加（通常 beam 会在结束前停止，但为安全）
                if full_seq[-1] != end_token:
                    full_seq.append(end_token)
                captions.append(full_seq)
                # ======================================================

            return captions


Writing models/decoder.py


In [9]:
%%writefile models/attention.py

import torch
from torch import Tensor, nn


class AdditiveAttention(nn.Module):
    """Additive attention in Show, Attend, and Tell."""

    def __init__(self, feature_dim: int, hidden_dim: int):
        """Initialize AdditiveAttention.

        Args:
            feature_dim (int): Feature dimension.
            hidden_dim (int): Hidden dimension.
        """
        super().__init__()
        # ======== TODO: Init linear layers for attention ========
        # Note on Features. Refer to the archived source for the detailed original comment.
        # Note on Output. Refer to the archived source for the detailed original comment.
        # ========================================================

    def forward(self, features: Tensor, hidden: Tensor):
        """Forward pass of AdditiveAttention.

        Args:
            features (Tensor): Input features with shape (batch, num_pixels, feature_dim).
            hidden (Tensor): Hidden state with shape (batch, hidden_dim).

        Returns:
            Tensor: Attention output with shape (batch, num_pixels, hidden_dim).
        """
        # ============ TODO: forward pass ============
        # e = v * tanh(Wf * features + Wh * hidden)
        # alpha = softmax(e)
        # context = sum(alpha * features)
        raise NotImplementedError
        # ============================================
        return context


Writing models/attention.py


In [10]:
%%writefile models/__init__.py
from .attention import AdditiveAttention
from .decoder import RNNDecoder, RNNDecoderWithAttention
from .encoder import Encoder

Writing models/__init__.py


### Training

This cell documents the image caption generation experiment. For the full original methodology, mathematical derivations, and commentary, see the archived notebook in `archive/original_notebooks/`.


In [11]:
"""训练模型。"""

import json
from dataclasses import asdict, dataclass
from datetime import datetime
from pathlib import Path

import torch
import torch.optim as optim
import torchvision.transforms as transforms
from nltk.translate.bleu_score import corpus_bleu
from torch import Tensor, nn
from torch.optim.lr_scheduler import ReduceLROnPlateau
from torch.utils.data import DataLoader, Subset
from tqdm import tqdm

from caption_dataset import CaptionDataset, collate_fn
from models import Encoder, RNNDecoder, RNNDecoderWithAttention
from utils import AverageMeter

Decoder = RNNDecoder | RNNDecoderWithAttention


@dataclass
class Config:
    """Training config."""

    batch_size: int = 32
    """Batch size for training and testing."""
    num_workers: int = 12
    """Number of dataset loading workers."""
    learning_rate: float = 5e-4
    """Learning rate (of decoder)."""
    encoder_learning_rate: float = 1e-4
    """Encoder learning rate, used only when finetune_encoder=True."""
    weight_decay: float = 1e-4
    """Weight decay."""
    num_epochs: int = 20
    """Number of training epochs."""
    embed_dim = 512
    """Dimension of word embedding."""
    hidden_dim = 1024
    """Dimension of decoder hidden state."""
    grad_clip: float = 5.0
    """Grad clip norm."""
    dropout: float = 0.2
    """Dropout rate."""
    max_decode_len: int = 30
    """Maximum decode length. Default: 30."""
    beam_size: int = 1
    """Beam search. 1 for greedy decoding. Default: 1."""
    device: str = "cuda"
    """Which device to use."""
    # TODO set directory to your own path
    dataset_dir: str = "/kaggle/input/datasets/adityajn105/flickr8k"
    preprocess_dir: str = "/kaggle/input/datasets/fdiiiiii/flickr8k"
    """Dataset directory."""
    output_dir: str = "output"
    """Output directory."""
    finetune_encoder: bool = False
    """Fine-tune encoder or not."""
    use_attention: bool = False
    """Use attention or not."""


def load_models(vocab_size: int, cfg: Config):
    """Load encoder and decoder.

    Args:
        vocab_size (int): Vocabulary size.
        cfg (Config): config.

    Returns:
        tuple[Module, Module]: Encoder and decoder.
    """
    device = cfg.device
    encoder = Encoder(encoded_size=(7, 7), finetune=cfg.finetune_encoder).to(device)

    print("Trying to forward one dummy tensor to get the feature dimension...")
    dummy_features: Tensor = encoder(torch.zeros(1, 3, 224, 224, device=device))
    feature_dim = dummy_features.shape[-1]
    print(f"Feature dimension: {feature_dim}")

    if cfg.use_attention:
        decoder = RNNDecoderWithAttention(
            cfg.embed_dim, cfg.hidden_dim, vocab_size, feature_dim, dropout=cfg.dropout
        ).to(device)
    else:
        decoder = RNNDecoder(
            cfg.embed_dim, cfg.hidden_dim, vocab_size, feature_dim, dropout=cfg.dropout
        ).to(device)
    return encoder, decoder


def train_one_epoch(
    epoch: int,
    encoder: Encoder,
    decoder: Decoder,
    dataloader: DataLoader,
    criterion: nn.Module,
    optimizers: list[optim.Optimizer],
    schedulers: list[ReduceLROnPlateau],
    cfg: Config,
):
    """Train the model for one epoch.

    Args:
    """
    device = cfg.device
    if cfg.finetune_encoder:
        encoder.train()
    else:
        encoder.eval()
    decoder.train()
    loss_meter = AverageMeter()
    acc_meter = AverageMeter()

    pbar = tqdm(dataloader, desc=f"Epoch {epoch + 1}/{cfg.num_epochs}")
    for batch_idx, (imgs, captions, lengths) in enumerate(pbar):
        imgs: Tensor
        captions: Tensor
        lengths: Tensor
        imgs = imgs.to(device)
        captions = captions.to(device)

        # 前向传播
        outputs: Tensor = decoder(encoder(imgs), captions)

        # Note on Compute. Refer to the archived source for the detailed original comment.
        # Note on Compute. Refer to the archived source for the detailed original comment.
        targets = captions[:, 1:].reshape(-1)  # 目标是输入的偏移一位
        outputs = outputs.reshape(-1, outputs.shape[2])
        loss: Tensor = criterion(outputs, targets)

        # 反向传播
        [o.zero_grad() for o in optimizers]
        loss.backward()

        # Note on Gradient. Refer to the archived source for the detailed original comment.
        torch.nn.utils.clip_grad_norm_(decoder.parameters(), max_norm=cfg.grad_clip)
        if cfg.finetune_encoder:
            torch.nn.utils.clip_grad_norm_(
                filter(lambda p: p.requires_grad, encoder.parameters()),
                max_norm=cfg.grad_clip,
            )

        # Note on Parameters. Refer to the archived source for the detailed original comment.
        [o.step() for o in optimizers]

        with torch.no_grad():
            targets_not_padding = targets != 0
            num_targets: int = targets_not_padding.long().sum().item()
            num_correct: int = (
                torch.logical_and(outputs.argmax(-1) == targets, targets_not_padding)
                .long()
                .sum()
                .item()
            )

        loss_meter.update(loss.item(), num_targets)
        acc_meter.update(num_correct / num_targets, num_targets)
        pbar.set_postfix(
            {"Loss": f"{loss_meter.avg:.4f}", "Acc": f"{acc_meter.avg:.1%}"}
        )

    # Note on Learning Rate. Refer to the archived source for the detailed original comment.
    [sched.step(loss_meter.avg) for sched in schedulers]

    return loss_meter.avg


def decode_caption(caption: list[int], rev_vocab: dict[int, str]) -> str:
    """Decode caption to sentence with vocabulary.

    Args:
        caption (list[int]): token ids.
        rev_vocab (dict[int, str]): map of token id to word.

    Returns:
        str: Decoded sentences.
    """
    words: list[str] = []
    for m in caption:
        s = rev_vocab.get(m, "<unk>")
        if s in ("<start>", "<end>"):
            continue
        words.append(s)
    sentence = " ".join(words) + "."
    return sentence.capitalize()


@torch.no_grad()
def evaluate(
    encoder: Encoder,
    decoder: Decoder,
    dataloader: DataLoader,
    vocab: dict[str, int],
    cfg: Config,
):
    """Evaluate the model (validation or test).

    Args:
        encoder (Encoder): Encoder model.
        decoder (Decoder): Decoder model.
        dataloader (DataLoader): DataLoader for evaluation.
        device (str): device name.
        vocab (dict[str, int]): map of token to token id.

    Returns:
        dict[str, float]: BLEU-1, BLEU-2, BLEU-3, BLEU-4 scores.
    """
    device = cfg.device
    encoder.eval()
    decoder.eval()
    hypotheses: list[list[int]] = []
    references: list[list[list[int]]] = []

    for imgs, captions, lengths, all_captions in tqdm(dataloader, desc="Evaluating"):
        all_captions: list[list[list[int]]]
        imgs: Tensor
        # 移到设备上
        imgs = imgs.to(device)

        # Strip <start> and <end>
        preds = decoder.generate_caption(
            encoder(imgs), vocab, max_len=cfg.max_decode_len, beam_size=cfg.beam_size
        )
        for i, pred in enumerate(preds):
            preds[i] = pred[1:-1]
        for i, caps in enumerate(all_captions):
            for j, cap in enumerate(caps):
                caps[j] = cap[1:-1]
        hypotheses.extend(preds)
        references.extend(all_captions)

    print("Calculating BLEU score...")
    # bleu1 = corpus_bleu(references, hypotheses, weights=(1.0, 0, 0, 0))
    # bleu2 = corpus_bleu(references, hypotheses, weights=(0.5, 0.5, 0, 0))
    # bleu3 = corpus_bleu(references, hypotheses, weights=(0.33, 0.33, 0.33, 0))
    bleu4 = corpus_bleu(references, hypotheses, weights=(0.25, 0.25, 0.25, 0.25))

    return {
        # "bleu-1": bleu1,
        # "bleu-2": bleu2,
        # "bleu-3": bleu3,
        "bleu-4": bleu4,
    }


def main():
    """
    主函数：训练和评估模型
    """
    # 配置
    cfg = Config()
    if not torch.cuda.is_available():
        cfg.device = "cpu"
        print("Warning: Using CPU for training.")
    print("Config:")
    print(json.dumps(asdict(cfg), indent=2, ensure_ascii=False))

    timestamp = datetime.now().strftime(r"%Y%m%d-%H%M%S")
    output_dir = Path(cfg.output_dir) / timestamp
    print(f"Output dir: {output_dir}")

    # Note on Training, Data. Refer to the archived source for the detailed original comment.
    train_transform = transforms.Compose(
        [
            # transforms.RandomResizedCrop(224),
            transforms.Resize((224, 224)),
            transforms.RandomHorizontalFlip(),
            transforms.ColorJitter(brightness=0.1, contrast=0.1, saturation=0.1),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
        ]
    )

    # Note on Validation, Data. Refer to the archived source for the detailed original comment.
    val_transform = transforms.Compose(
        [
            # transforms.Resize(224),
            # transforms.CenterCrop(224),
            transforms.Resize((224, 224)),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
        ]
    )

    # Note on Data Loading, Dataset, Create, Load. Refer to the archived source for the detailed original comment.
    # Train images cannot be cached
    train_dataset = CaptionDataset(
        cfg.dataset_dir, preprocess_dir=cfg.preprocess_dir, split="train", transform=train_transform, cache=False
    )
    vocab = train_dataset.vocab
    vocab_size = len(vocab)
    val_dataset = CaptionDataset(
        cfg.dataset_dir, preprocess_dir=cfg.preprocess_dir, split="val", transform=val_transform, cache=False
    )
    print(
        f"Dataset: {train_dataset.split} {len(train_dataset):,} captions, "
        f"{val_dataset.split} {len(val_dataset):,} captions"
    )

    train_loader = DataLoader(
        train_dataset,
        batch_size=cfg.batch_size,
        shuffle=True,
        num_workers=cfg.num_workers,
        collate_fn=collate_fn,
        pin_memory=True,
    )

    val_loader = DataLoader(
        val_dataset,
        batch_size=cfg.batch_size,
        shuffle=False,
        num_workers=4,
        collate_fn=collate_fn,
        pin_memory=True,
    )

    # Note on Model, Create. Refer to the archived source for the detailed original comment.
    device = cfg.device
    encoder, decoder = load_models(vocab_size, cfg)

    # Note on Loss Function, Define, Optimizer. Refer to the archived source for the detailed original comment.
    criterion = nn.CrossEntropyLoss(ignore_index=0)  # Ignore <pad>
    optimizers = [
        optim.Adam(
            decoder.parameters(), lr=cfg.learning_rate, weight_decay=cfg.weight_decay
        )
    ]
    if cfg.finetune_encoder:
        optimizers.append(
            optim.Adam(
                filter(lambda p: p.requires_grad, encoder.parameters()),
                lr=cfg.encoder_learning_rate,
                weight_decay=1e-5,
            )
        )
    schedulers = [
        ReduceLROnPlateau(optimizer, factor=0.5, patience=5) for optimizer in optimizers
    ]

    # Note on Training, Loop. Refer to the archived source for the detailed original comment.
    best_bleu4 = 0.0

    for epoch in range(cfg.num_epochs):
        # Note on Training. Refer to the archived source for the detailed original comment.
        train_loss = train_one_epoch(
            epoch,
            encoder,
            decoder,
            train_loader,
            criterion,
            optimizers,
            schedulers,
            cfg,
        )
        print(f"Train loss: {train_loss:.4f}")
        # Note on Validation. Refer to the archived source for the detailed original comment.
        scores = evaluate(encoder, decoder, val_loader, vocab, cfg)
        bleu4 = scores["bleu-4"]
        print(f"BLEU-4: {bleu4:.4f}")

        # Note on Model, Save. Refer to the archived source for the detailed original comment.
        if bleu4 > best_bleu4:
            best_bleu4 = bleu4
            if not output_dir.exists():
                output_dir.mkdir(parents=True, exist_ok=True)
                # Dump config
                with open(output_dir / "config.json", "w") as f:
                    json.dump(asdict(cfg), f, indent=2, ensure_ascii=True)
            torch.save(
                {
                    "config": asdict(cfg),
                    "encoder": encoder.state_dict(),
                    "decoder": decoder.state_dict(),
                },
                output_dir / "best_model.pt",
            )
            print(
                f"Best model saved to: {output_dir / 'best_model.pt'}  (epoch: {epoch + 1}, BLEU-4: {best_bleu4:.4f})"
            )


if __name__ == "__main__":
    main()

Config:
{
  "batch_size": 32,
  "num_workers": 12,
  "learning_rate": 0.0005,
  "encoder_learning_rate": 0.0001,
  "weight_decay": 0.0001,
  "num_epochs": 20,
  "grad_clip": 5.0,
  "dropout": 0.2,
  "max_decode_len": 30,
  "beam_size": 1,
  "device": "cuda",
  "dataset_dir": "/kaggle/input/datasets/adityajn105/flickr8k",
  "preprocess_dir": "/kaggle/input/datasets/fdiiiiii/flickr8k",
  "output_dir": "output",
  "finetune_encoder": false,
  "use_attention": false
}
Output dir: output/20260731-105220
Dataset: train 30,000 captions, val 5,000 captions


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 12 worker processes in total. Our suggested max number of worker in current system is 4, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()


Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /root/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 189MB/s]


Trying to forward one dummy tensor to get the feature dimension...
Feature dimension: 2048


Epoch 1/20:   0%|          | 0/938 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:432: UserWarning: This DataLoader will create 12 worker processes in total. Our suggested max number of worker in current system is 4, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()
Epoch 1/20: 100%|██████████| 938/938 [02:06<00:00,  7.42it/s, Loss=3.6045, Acc=33.2%]


... (75 outputs left out)

Epoch 20/20: 100%|██████████| 938/938 [02:07<00:00,  7.36it/s, Loss=1.5320, Acc=59.0%]


Train loss: 1.5320


Evaluating: 100%|██████████| 157/157 [00:18<00:00,  8.49it/s]


Calculating BLEU score...
BLEU-4: 0.1652
